In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

url = "https://github.com/devtlv/Datasets-GEN-AI-Bootcamp/raw/refs/heads/main/Week%205/Day%204%20-%20Statistics%20for%20Machine%20Learning/Heart%20Disease%20Prediction%20Dataset.zip"
df = pd.read_csv(url, compression='zip')

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())
df.head()

Shape: (270, 14)

Columns: ['age', 'sex ', 'chest pain type', 'resting blood pressure', 'serum cholestoral', 'fasting blood sugar', 'resting electrocardiographic results', 'max heart rate', 'exercise induced angina', 'oldpeak', 'ST segment', 'major vessels', 'thal', 'heart disease']


,age,sex,chest pain type,resting blood pressure,serum cholestoral,fasting blood sugar,resting electrocardiographic results,max heart rate,exercise induced angina,oldpeak,ST segment,major vessels,thal,heart disease
0,70,1,4,130,322,0,2,109,0,2.4,2,3,3,2
1,67,0,3,115,564,0,2,160,0,1.6,2,0,7,1
2,57,1,2,124,261,0,0,141,0,0.3,1,0,7,2
3,64,1,4,128,263,0,0,105,1,0.2,2,1,7,1
4,74,0,2,120,269,0,2,121,1,0.2,1,1,3,1


# Week 5 Day 2 — Exercises XP: Heart Disease Prediction with Hyperparameter Tuning

## Exercise 1: Exploratory Data Analysis

In [2]:
print("Missing values:", df.isnull().sum().sum())
print("\nTarget variable unique values:", df['heart disease'].unique())
print("\nTarget distribution:")
print(df['heart disease'].value_counts())
print("\nData types:\n", df.dtypes)

Missing values: 0

Target variable unique values: [2 1]

Target distribution:
heart disease
1    150
2    120
Name: count, dtype: int64

Data types:
 age                                       int64
sex                                       int64
chest pain type                           int64
resting blood pressure                    int64
serum cholestoral                         int64
fasting blood sugar                       int64
resting electrocardiographic results      int64
max heart rate                            int64
exercise induced angina                   int64
oldpeak                                 float64
ST segment                                int64
major vessels                             int64
thal                                      int64
heart disease                             int64
dtype: object


Data is fully clean (0 missing values, all numeric already). The target `heart disease` uses 1/2 encoding rather than the usual 0/1 (1 = disease present, 2 = no disease) — worth noting explicitly to avoid misinterpretation later. Class balance is reasonable (150 vs. 120, ~55%/45%), unlike the severe imbalance in the earlier Diabetes exercise — meaning accuracy will be a more trustworthy metric here without needing heavy reliance on precision/recall to catch a hidden problem.

In [3]:
from sklearn.model_selection import train_test_split

X = df.drop('heart disease', axis=1)
y = df['heart disease']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)


Train shape: (216, 13)
Test shape: (54, 13)


## Exercise 2: Logistic Regression without Grid Search

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train, y_train)

log_pred = log_model.predict(X_test)
log_accuracy = accuracy_score(y_test, log_pred)

print(f"Logistic Regression Accuracy (default settings): {log_accuracy:.4f} ({log_accuracy*100:.2f}%)")

Logistic Regression Accuracy (default settings): 0.8519 (85.19%)


C:\Users\Frenki\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [5]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

log_model = LogisticRegression(max_iter=1000)
log_model.fit(X_train_scaled, y_train)

log_pred = log_model.predict(X_test_scaled)
log_accuracy = accuracy_score(y_test, log_pred)

print(f"Logistic Regression Accuracy (scaled data): {log_accuracy:.4f} ({log_accuracy*100:.2f}%)")

Logistic Regression Accuracy (scaled data): 0.8519 (85.19%)


## Exercise 3: Logistic Regression with Grid Search

In [6]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'C': [0.01, 0.1, 1, 10, 100],
    'penalty': ['l1', 'l2'],
    'solver': ['liblinear']
}

grid_search = GridSearchCV(LogisticRegression(max_iter=1000), param_grid, cv=5)
grid_search.fit(X_train_scaled, y_train)

print("Best parameters:", grid_search.best_params_)
print("Best cross-validation score:", grid_search.best_score_)

best_log_model = grid_search.best_estimator_
grid_pred = best_log_model.predict(X_test_scaled)
grid_accuracy = accuracy_score(y_test, grid_pred)

print(f"\nTuned Logistic Regression Accuracy: {grid_accuracy:.4f} ({grid_accuracy*100:.2f}%)")
print(f"Baseline (no tuning) Accuracy: 0.8519 (85.19%)")

Best parameters: {'C': 0.1, 'penalty': 'l2', 'solver': 'liblinear'}
Best cross-validation score: 0.8426004228329809

Tuned Logistic Regression Accuracy: 0.8519 (85.19%)
Baseline (no tuning) Accuracy: 0.8519 (85.19%)


C:\Users\Frenki\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
C:\Users\Frenki\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:1429: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
C:\Users\Frenki\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\linear_model\_logistic.py:1403: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default

Tuning did NOT improve Logistic Regression on this dataset — both tuned and untuned versions scored identically at 85.19%. GridSearch's "best" combination (C=0.1, penalty='l2') matched but did not exceed the default settings. With only 270 total samples, there may simply be insufficient data for hyperparameter differences to meaningfully change the outcome — echoing the Day 1 finding that added complexity/tuning doesn't automatically improve results.

## Exercise 4: SVM without Grid Search

In [7]:
from sklearn.svm import SVC

svm_model = SVC(kernel='rbf')
svm_model.fit(X_train_scaled, y_train)

svm_pred = svm_model.predict(X_test_scaled)
svm_accuracy = accuracy_score(y_test, svm_pred)

print(f"SVM Accuracy (default, rbf kernel): {svm_accuracy:.4f} ({svm_accuracy*100:.2f}%)")
print(f"Logistic Regression baseline: 0.8519 (85.19%)")

SVM Accuracy (default, rbf kernel): 0.8148 (81.48%)
Logistic Regression baseline: 0.8519 (85.19%)


SVM with default settings (81.48%) performed WORSE than Logistic Regression's baseline (85.19%) — a useful reminder that a more complex-sounding algorithm isn't automatically better; its performance depends heavily on choosing appropriate settings for the specific dataset. Unlike Logistic Regression (which showed no room for improvement from tuning), SVM has more hyperparameters (C, kernel, gamma) that could plausibly be poorly matched to this dataset by default — making Exercise 5's tuning step a meaningful test of whether SVM can actually be competitive here.

## Exercise 5: SVM with Grid Search

In [8]:
svm_param_grid = {
    'C': [0.1, 1, 10, 100],
    'kernel': ['linear', 'rbf'],
    'gamma': ['scale', 'auto']
}

svm_grid_search = GridSearchCV(SVC(), svm_param_grid, cv=5)
svm_grid_search.fit(X_train_scaled, y_train)

print("Best parameters:", svm_grid_search.best_params_)
print("Best cross-validation score:", svm_grid_search.best_score_)

best_svm_model = svm_grid_search.best_estimator_
svm_grid_pred = best_svm_model.predict(X_test_scaled)
svm_grid_accuracy = accuracy_score(y_test, svm_grid_pred)

print(f"\nTuned SVM Accuracy: {svm_grid_accuracy:.4f} ({svm_grid_accuracy*100:.2f}%)")
print(f"Untuned SVM Accuracy: 0.8148 (81.48%)")
print(f"Logistic Regression baseline: 0.8519 (85.19%)")

Best parameters: {'C': 0.1, 'gamma': 'scale', 'kernel': 'linear'}
Best cross-validation score: 0.8380549682875265

Tuned SVM Accuracy: 0.8519 (85.19%)
Untuned SVM Accuracy: 0.8148 (81.48%)
Logistic Regression baseline: 0.8519 (85.19%)


Tuning improved SVM significantly — from 81.48% (default) to 85.19% (tuned), a real 3.7 percentage point gain, now matching Logistic Regression's score exactly. Notably, GridSearch selected a linear kernel as best, not the rbf kernel manually guessed in Exercise 4 — proving that the systematic search found a better-suited setting than human intuition did. This is a direct, concrete illustration of Day 2's core lesson: hyperparameter tuning can meaningfully rescue an underperforming model, unlike Logistic Regression's case where tuning had genuinely nothing to improve.

In [9]:
from xgboost import XGBClassifier

In [10]:
pip install xgboost

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\Frenki\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [11]:
from xgboost import XGBClassifier

In [12]:
y_train_xgb = y_train - 1
y_test_xgb = y_test - 1

print("Original labels:", y_train.unique())
print("Remapped labels:", y_train_xgb.unique())

Original labels: [1 2]
Remapped labels: [0 1]


In [13]:
xgb_model = XGBClassifier(eval_metric='logloss')
xgb_model.fit(X_train_scaled, y_train_xgb)

xgb_pred = xgb_model.predict(X_test_scaled)
xgb_accuracy = accuracy_score(y_test_xgb, xgb_pred)

print(f"XGBoost Accuracy (default settings): {xgb_accuracy:.4f} ({xgb_accuracy*100:.2f}%)")
print(f"Logistic Regression baseline: 0.8519 (85.19%)")
print(f"Tuned SVM: 0.8519 (85.19%)")

XGBoost Accuracy (default settings): 0.8148 (81.48%)
Logistic Regression baseline: 0.8519 (85.19%)
Tuned SVM: 0.8519 (85.19%)


XGBoost with default settings scored 81.48%, matching untuned SVM's score exactly, and both underperform Logistic Regression's 85.19%. This continues the pattern from the SVM comparison: more configurable algorithms (SVM, XGBoost) appear to need tuning to reach competitive performance on this small dataset, while Logistic Regression's defaults were already well-suited. Note: XGBoost required remapping the target labels from {1,2} to {0,1}, since it strictly requires class labels to start at 0.

## Exercise 7: XGBoost with Grid Search

In [14]:
xgb_param_grid = {
    'learning_rate': [0.01, 0.1, 0.2],
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 7]
}

xgb_grid_search = GridSearchCV(XGBClassifier(eval_metric='logloss'), xgb_param_grid, cv=5)
xgb_grid_search.fit(X_train_scaled, y_train_xgb)

print("Best parameters:", xgb_grid_search.best_params_)
print("Best cross-validation score:", xgb_grid_search.best_score_)

best_xgb_model = xgb_grid_search.best_estimator_
xgb_grid_pred = best_xgb_model.predict(X_test_scaled)
xgb_grid_accuracy = accuracy_score(y_test_xgb, xgb_grid_pred)

print(f"\nTuned XGBoost Accuracy: {xgb_grid_accuracy:.4f} ({xgb_grid_accuracy*100:.2f}%)")
print(f"Untuned XGBoost: 0.8148 (81.48%)")
print(f"Logistic Regression baseline: 0.8519 (85.19%)")
print(f"Tuned SVM: 0.8519 (85.19%)")

Best parameters: {'learning_rate': 0.2, 'max_depth': 3, 'n_estimators': 100}
Best cross-validation score: 0.8195560253699788

Tuned XGBoost Accuracy: 0.8333 (83.33%)
Untuned XGBoost: 0.8148 (81.48%)
Logistic Regression baseline: 0.8519 (85.19%)
Tuned SVM: 0.8519 (85.19%)


## Final Comparison and Conclusion

| Model | Untuned | Tuned |
|---|---|---|
| Logistic Regression | 85.19% | 85.19% (no change) |
| SVM | 81.48% | 85.19% (+3.7 points) |
| XGBoost | 81.48% | 83.33% (+1.85 points) |

**Key takeaway:** on this small (270-patient), reasonably-balanced dataset, the simplest model (Logistic Regression) performed best overall, and required no tuning to get there. Tuning meaningfully helped SVM (fully closing its gap with Logistic Regression) and helped XGBoost partially, but XGBoost — often considered the most powerful of these three algorithms in general — never caught up to the simpler models here. This is a genuinely important, realistic lesson: a more sophisticated algorithm is not automatically better, especially on small datasets, and systematic tuning (GridSearchCV) can meaningfully close performance gaps but isn't guaranteed to make every model the best choice.